# Cholec80 — standardized recognition Transformer

This notebook preserves the uploaded Transformer's 64-frame temporal-window
design while fixing its experiment/output protocol. It runs **teammates**,
**finetuned**, and **original** features and creates one SF-TMN/recurrent-compatible
archive per source.

Important corrections:

- source-specific result directories and checkpoint names prevent overwrites;
- overlapping inference windows cover every frame, including each video's tail;
- checkpoint/loss/post-processing selection uses validation segmental F1 only;
- test inference occurs once after the model and post-processing are frozen;
- checkpoints, per-video log-probabilities, histories, protocol, and metrics are saved.

Legacy Transformer checkpoints can be mapped in `PRETRAINED_CHECKPOINTS`; the
notebook will then rebuild standardized artifacts without retraining.

In [1]:
import gc
import json
import math
import random
import shutil
import time
import warnings
import zipfile
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    jaccard_score,
    precision_score,
    recall_score,
)

warnings.filterwarnings("ignore")

SEED = 42
TRAIN_IDS = list(range(1, 33))
VAL_IDS = list(range(33, 41))
TEST_IDS = list(range(41, 81))
NUM_CLASSES, FEATURE_DIM, FPS = 7, 2048, 1
PHASE_NAMES = [
    "Preparation", "CalotTriangleDissection", "ClippingCutting",
    "GallbladderDissection", "GallbladderPackaging",
    "CleaningCoagulation", "GallbladderRetraction",
]

# All three sources are required. Attach them to the Kaggle notebook before running.
FEATURE_SOURCES = {
    "teammates": {
        "description": "Giorgia/Marta 1 FPS frames -> ImageNet ResNet50 features",
        "paths": [
            "/kaggle/input/datasets/christiancorvino/features-teammates/features",
            "/kaggle/input/features-teammates/features",
        ],
    },
    "finetuned": {
        "description": "Christian fine-tuned ResNet50 features from original videos",
        "paths": [
            "/kaggle/input/datasets/christiancorvino/features-resnet50finetuned/features_finetuned",
            "/kaggle/input/features-resnet50finetuned/features_finetuned",
            "/kaggle/input/notebooks/christiancorvino/cholec80-resnet50-finetuning/features_finetuned",
        ],
    },
    "original": {
        "description": "Christian initial direct-from-video ImageNet ResNet50 features",
        "paths": [
            "/kaggle/input/datasets/christiancorvino/cholec80-features/features",
            "/kaggle/input/cholec80-features/features",
        ],
    },
}

# Optional recovery mode. Map "feature_source:candidate" to an extracted legacy
# checkpoint. A mapped checkpoint is evaluated and standardized without retraining.
# Example:
# PRETRAINED_CHECKPOINTS = {
#     "teammates:weighted_ce": "/kaggle/input/legacy-run/best_model_weighted_ce.pt",
# }
PRETRAINED_CHECKPOINTS = {}
TRAIN_IF_CHECKPOINT_MISSING = True

OUTPUT_ROOT = Path("/kaggle/working")
NORMALIZE_FEATURES = False
USE_AMP = True
USE_VITERBI_CANDIDATE = True

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running this notebook.")
DEVICE = torch.device("cuda")
USE_AMP = bool(USE_AMP and DEVICE.type == "cuda")
torch.set_float32_matmul_precision("high")

def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

reset_seed()
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def resolve_feature_directories():
    resolved, missing = {}, {}
    for source, info in FEATURE_SOURCES.items():
        directory = next((Path(p) for p in info["paths"] if Path(p).is_dir()), None)
        if directory is None:
            missing[source] = info["paths"]
        else:
            resolved[source] = directory
    if missing:
        details = "\n".join(
            f"- {source}: " + " or ".join(paths) for source, paths in missing.items()
        )
        raise FileNotFoundError(
            "Attach all three feature datasets. Missing sources:\n" + details
        )
    return resolved

RESOLVED_FEATURE_DIRS = resolve_feature_directories()
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch / CUDA:", torch.__version__, "/", torch.version.cuda)
print("Feature sources:", {k: str(v) for k, v in RESOLVED_FEATURE_DIRS.items()})
print("Split: 32 train / 8 validation / 40 test")


from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

GPU: Tesla T4
PyTorch / CUDA: 2.10.0+cu128 / 12.8
Feature sources: {'teammates': '/kaggle/input/datasets/christiancorvino/features-teammates/features', 'finetuned': '/kaggle/input/datasets/christiancorvino/features-resnet50finetuned/features_finetuned', 'original': '/kaggle/input/datasets/christiancorvino/cholec80-features/features'}
Split: 32 train / 8 validation / 40 test


## Metrics and validation-only post-processing

In [2]:
def extract_segments(sequence):
    sequence = np.asarray(sequence, dtype=np.int64)
    if len(sequence) == 0:
        return np.array([], int), np.array([], int), np.array([], int)
    change = np.r_[True, sequence[1:] != sequence[:-1]]
    starts = np.flatnonzero(change)
    ends = np.r_[starts[1:], len(sequence)]
    return sequence[starts], starts, ends

def normalized_edit_score(target, prediction):
    target_segments = extract_segments(target)[0].tolist()
    pred_segments = extract_segments(prediction)[0].tolist()
    previous = list(range(len(pred_segments) + 1))
    for i, target_label in enumerate(target_segments, 1):
        current = [i]
        for j, pred_label in enumerate(pred_segments, 1):
            current.append(min(
                current[-1] + 1,
                previous[j] + 1,
                previous[j - 1] + int(target_label != pred_label),
            ))
        previous = current
    distance = previous[-1]
    return 1.0 - distance / max(len(target_segments), len(pred_segments), 1)

def segmental_f1_counts(target, prediction, overlap):
    gt_labels, gt_starts, gt_ends = extract_segments(target)
    pr_labels, pr_starts, pr_ends = extract_segments(prediction)
    matched = np.zeros(len(gt_labels), dtype=bool)
    tp = fp = 0
    for label, start, end in zip(pr_labels, pr_starts, pr_ends):
        intersection = np.minimum(end, gt_ends) - np.maximum(start, gt_starts)
        union = np.maximum(end, gt_ends) - np.minimum(start, gt_starts)
        iou = (intersection / np.maximum(union, 1)) * (gt_labels == label)
        best = int(np.argmax(iou)) if len(iou) else -1
        if best >= 0 and iou[best] >= overlap and not matched[best]:
            tp += 1
            matched[best] = True
        else:
            fp += 1
    return tp, fp, int((~matched).sum())

def frame_metrics(target, prediction):
    labels = np.arange(NUM_CLASSES)
    return {
        "accuracy": accuracy_score(target, prediction),
        "precision_macro": precision_score(
            target, prediction, labels=labels, average="macro", zero_division=0),
        "recall_macro": recall_score(
            target, prediction, labels=labels, average="macro", zero_division=0),
        "f1_macro": f1_score(
            target, prediction, labels=labels, average="macro", zero_division=0),
        "jaccard_macro": jaccard_score(
            target, prediction, labels=labels, average="macro", zero_division=0),
    }

def evaluate_sequences(results, prediction_key="preds"):
    all_target, all_prediction, edits = [], [], []
    counts = {0.10: [0, 0, 0], 0.25: [0, 0, 0], 0.50: [0, 0, 0]}
    per_video_frames = []
    for video_id in sorted(results):
        target = np.asarray(results[video_id]["labels"])
        prediction = np.asarray(results[video_id][prediction_key])
        all_target.append(target)
        all_prediction.append(prediction)
        edits.append(normalized_edit_score(target, prediction))
        per_video_frames.append(frame_metrics(target, prediction))
        for overlap in counts:
            values = segmental_f1_counts(target, prediction, overlap)
            counts[overlap] = [a + b for a, b in zip(counts[overlap], values)]

    target = np.concatenate(all_target)
    prediction = np.concatenate(all_prediction)
    metrics = frame_metrics(target, prediction)
    metrics["edit"] = float(np.mean(edits))
    for overlap, (tp, fp, fn) in counts.items():
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        metrics[f"f1_at_{int(overlap * 100)}"] = (
            2 * precision * recall / max(precision + recall, 1e-12)
        )
    metrics["f1_segmental_avg"] = float(np.mean([
        metrics["f1_at_10"], metrics["f1_at_25"], metrics["f1_at_50"]
    ]))

    for metric_name in ("accuracy", "precision_macro", "recall_macro", "jaccard_macro"):
        values = np.array([row[metric_name] for row in per_video_frames])
        metrics[f"video_mean_{metric_name}"] = float(values.mean())
        metrics[f"video_std_{metric_name}"] = float(values.std(ddof=0))
    return metrics

def per_video_metrics(results, prediction_key="preds"):
    rows = []
    for video_id in sorted(results):
        target = results[video_id]["labels"]
        prediction = results[video_id][prediction_key]
        frame = frame_metrics(target, prediction)
        segmental = {}
        for overlap in (0.10, 0.25, 0.50):
            tp, fp, fn = segmental_f1_counts(target, prediction, overlap)
            precision = tp / max(tp + fp, 1)
            recall = tp / max(tp + fn, 1)
            segmental[f"f1_at_{int(overlap * 100)}"] = (
                2 * precision * recall / max(precision + recall, 1e-12)
            )
        rows.append({
            "video": video_id,
            **frame,
            "edit": normalized_edit_score(target, prediction),
            **segmental,
            "f1_segmental_avg": float(np.mean(list(segmental.values()))),
        })
    return pd.DataFrame(rows)

def training_transition_model(dataset):
    transitions = np.full((NUM_CLASSES, NUM_CLASSES), 1e-3, dtype=np.float64)
    starts = np.full(NUM_CLASSES, 1e-3, dtype=np.float64)
    for item in dataset.data:
        labels = item["labels"].numpy()
        starts[labels[0]] += 1
        for previous, current in zip(labels[:-1], labels[1:]):
            transitions[previous, current] += 1
    transitions /= transitions.sum(axis=1, keepdims=True)
    starts /= starts.sum()
    return np.log(transitions), np.log(starts)

def viterbi_decode(log_probabilities, transition_log, start_log):
    length, classes = log_probabilities.shape
    score = np.full((length, classes), -np.inf, dtype=np.float64)
    back = np.zeros((length, classes), dtype=np.int64)
    score[0] = start_log + log_probabilities[0]
    for time_index in range(1, length):
        candidates = score[time_index - 1][:, None] + transition_log
        back[time_index] = candidates.argmax(axis=0)
        score[time_index] = candidates.max(axis=0) + log_probabilities[time_index]
    path = np.zeros(length, dtype=np.int64)
    path[-1] = score[-1].argmax()
    for time_index in range(length - 2, -1, -1):
        path[time_index] = back[time_index + 1, path[time_index + 1]]
    return path

def add_postprocessing(results, transition_log, start_log):
    output = deepcopy(results)
    for item in output.values():
        item["viterbi"] = viterbi_decode(
            item["log_probs"], transition_log, start_log
        )
    return output

def validation_postprocessing_table(results):
    rows = [{"postprocessing": "raw", **evaluate_sequences(results, "preds")}]
    if USE_VITERBI_CANDIDATE:
        rows.append({
            "postprocessing": "viterbi",
            **evaluate_sequences(results, "viterbi"),
        })
    return pd.DataFrame(rows).sort_values(
        ["f1_segmental_avg", "edit", "f1_macro"], ascending=False
    ).reset_index(drop=True)

def load_checkpoint_state(path):
    try:
        payload = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        payload = torch.load(path, map_location="cpu")
    if isinstance(payload, dict) and "model_state_dict" in payload:
        state = payload["model_state_dict"]
    else:
        state = payload
    return {
        (key[7:] if key.startswith("module.") else key): value
        for key, value in state.items()
    }

def cpu_state_dict(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

def plot_training_history(history, path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    if len(history):
        axes[0].plot(history["epoch"], history["train_loss"], label="train")
        axes[0].plot(history["epoch"], history["val_loss"], label="validation")
        axes[1].plot(history["epoch"], history["val_f1_segmental_avg"], label="F1 avg")
        axes[1].plot(history["epoch"], history["val_edit"], label="Edit")
        axes[1].plot(history["epoch"], history["val_accuracy"], label="Accuracy")
    axes[0].set(title="Loss", xlabel="Epoch")
    axes[1].set(title="Validation metrics", xlabel="Epoch", ylim=(0, 1))
    axes[0].legend()
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)

def save_confusion_matrix(results, prediction_key, title, path):
    target = np.concatenate([results[v]["labels"] for v in sorted(results)])
    prediction = np.concatenate([results[v][prediction_key] for v in sorted(results)])
    matrix = confusion_matrix(
        target, prediction, labels=np.arange(NUM_CLASSES), normalize="true"
    )
    fig, ax = plt.subplots(figsize=(9, 8))
    image = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(NUM_CLASSES), PHASE_NAMES, rotation=55, ha="right")
    ax.set_yticks(range(NUM_CLASSES), PHASE_NAMES)
    ax.set_xlabel("Predicted phase")
    ax.set_ylabel("True phase")
    ax.set_title(title)
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)

def make_archive(results_dir, run_name):
    zip_path = OUTPUT_ROOT / f"{run_name}_results.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in results_dir.rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to(results_dir.parent))
    return zip_path

## Feature sequences and temporal-window datasets

In [3]:
def feature_paths(feature_dir, video_id):
    return (
        feature_dir / f"video{video_id:02d}_features.npy",
        feature_dir / f"video{video_id:02d}_labels.npy",
    )

def inspect_feature_source(feature_dir):
    inventory = []
    for video_id in range(1, 81):
        feature_path, label_path = feature_paths(feature_dir, video_id)
        if not feature_path.exists() or not label_path.exists():
            raise FileNotFoundError(f"Missing feature/label pair for video {video_id:02d}")
        features = np.load(feature_path, mmap_mode="r")
        labels = np.load(label_path, mmap_mode="r")
        if features.ndim != 2 or features.shape[1] != FEATURE_DIM:
            raise ValueError(f"video{video_id:02d}: invalid feature shape {features.shape}")
        if labels.ndim != 1 or len(features) != len(labels):
            raise ValueError(
                f"video{video_id:02d}: unaligned shapes {features.shape}, {labels.shape}"
            )
        if not np.isfinite(features).all():
            raise ValueError(f"video{video_id:02d}: NaN or infinity in features")
        if labels.min() < 0 or labels.max() >= NUM_CLASSES:
            raise ValueError(f"video{video_id:02d}: invalid phase IDs")
        inventory.append({
            "video": video_id,
            "samples": len(labels),
            "minutes": len(labels) / (FPS * 60),
            "feature_dim": features.shape[1],
            "phases_present": len(np.unique(labels)),
        })
    return pd.DataFrame(inventory)

def calculate_train_stats(feature_dir, video_ids):
    total = 0
    sum_x = np.zeros(FEATURE_DIM, dtype=np.float64)
    sum_x2 = np.zeros(FEATURE_DIM, dtype=np.float64)
    for video_id in video_ids:
        x = np.asarray(
            np.load(feature_paths(feature_dir, video_id)[0], mmap_mode="r"),
            dtype=np.float64,
        )
        total += len(x)
        sum_x += x.sum(axis=0)
        sum_x2 += np.square(x).sum(axis=0)
    mean = sum_x / total
    std = np.sqrt(np.maximum(sum_x2 / total - np.square(mean), 1e-8))
    return mean.astype(np.float32), std.astype(np.float32)

class FeatureSequenceDataset:
    def __init__(self, feature_dir, video_ids, feature_mean=None, feature_std=None):
        self.video_ids = list(video_ids)
        self.data = []
        for video_id in self.video_ids:
            feature_path, label_path = feature_paths(feature_dir, video_id)
            x = np.load(feature_path).astype(np.float32)
            y = np.load(label_path).astype(np.int64)
            if feature_mean is not None:
                x = (x - feature_mean) / feature_std
            self.data.append({
                "video_id": video_id,
                "features": torch.from_numpy(x),
                "labels": torch.from_numpy(y),
            })

    def __len__(self):
        return len(self.data)

def balanced_class_weights(dataset):
    labels = np.concatenate([item["labels"].numpy() for item in dataset.data])
    counts = np.bincount(labels, minlength=NUM_CLASSES)
    values = len(labels) / (NUM_CLASSES * np.maximum(counts, 1))
    return torch.tensor(values, dtype=torch.float32, device=DEVICE), counts


WINDOW_LENGTH = 64
TRAIN_WINDOW_STRIDE = 1
INFERENCE_WINDOW_STRIDE = 32
BATCH_SIZE = 32
NUM_WORKERS = 2

class TransformerWindowDataset(Dataset):
    def __init__(self, sequence_dataset, train=True):
        self.sequence_dataset = sequence_dataset
        self.train = train
        self.samples = []
        for item_index, item in enumerate(sequence_dataset.data):
            length = len(item["labels"])
            if length < WINDOW_LENGTH:
                self.samples.append((item_index, 0))
            else:
                self.samples.extend(
                    (item_index, start)
                    for start in range(
                        0, length - WINDOW_LENGTH + 1,
                        TRAIN_WINDOW_STRIDE if train else WINDOW_LENGTH,
                    )
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        item_index, start = self.samples[index]
        item = self.sequence_dataset.data[item_index]
        features = item["features"].numpy()
        labels = item["labels"].numpy()
        speed = random.choice([0.9, 1.0, 1.1]) if self.train else 1.0
        requested = max(1, int(round(WINDOW_LENGTH * speed)))
        end = min(start + requested, len(labels))
        window_features = features[start:end]
        window_labels = labels[start:end]
        indices = np.linspace(
            0, len(window_labels) - 1, WINDOW_LENGTH
        ).astype(np.int64)
        return (
            torch.from_numpy(window_features[indices]).float(),
            torch.from_numpy(window_labels[indices]).long(),
        )

def window_sample_weights(window_dataset, class_weights_cpu):
    values = []
    for item_index, start in window_dataset.samples:
        labels = window_dataset.sequence_dataset.data[item_index]["labels"]
        center = min(start + WINDOW_LENGTH // 2, len(labels) - 1)
        values.append(float(class_weights_cpu[int(labels[center])]))
    return values

## Loss candidates

In [4]:
LOSS_CANDIDATES = [
    {"slug": "weighted_ce", "kind": "weighted_ce", "label_smoothing": 0.0},
    {"slug": "focal_g2", "kind": "focal", "gamma": 2.0, "label_smoothing": 0.1},
    {"slug": "boundary_aware", "kind": "boundary", "radius": 5,
     "boundary_weight": 3.0, "label_smoothing": 0.0},
]

class TransformerWeightedCE(nn.Module):
    def __init__(self, class_weights=None, label_smoothing=0.0):
        super().__init__()
        self.class_weights = class_weights
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        return F.cross_entropy(
            logits.transpose(1, 2), targets, weight=self.class_weights,
            label_smoothing=self.label_smoothing,
        )

class TransformerFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(
            logits.transpose(1, 2), targets, weight=self.alpha,
            reduction="none", label_smoothing=self.label_smoothing,
        )
        probability = torch.exp(-ce)
        return (((1 - probability) ** self.gamma) * ce).mean()

class TransformerBoundaryLoss(nn.Module):
    def __init__(self, class_weights=None, radius=5, boundary_weight=3.0,
                 label_smoothing=0.0):
        super().__init__()
        self.class_weights = class_weights
        self.radius = radius
        self.boundary_weight = boundary_weight
        self.label_smoothing = label_smoothing

    def boundary_weights(self, labels):
        batch, length = labels.shape
        changes = (labels[:, 1:] != labels[:, :-1]).float()
        signal = torch.zeros(batch, length, device=labels.device)
        signal[:, :-1] = torch.maximum(signal[:, :-1], changes)
        signal[:, 1:] = torch.maximum(signal[:, 1:], changes)
        kernel_size = 2 * self.radius + 1
        kernel = 1 - torch.arange(
            -self.radius, self.radius + 1, device=labels.device
        ).abs().float() / (self.radius + 1)
        smoothed = F.conv1d(
            signal.unsqueeze(1), kernel.view(1, 1, kernel_size), padding=self.radius
        ).squeeze(1)
        return 1.0 + smoothed.clamp(max=1.0) * self.boundary_weight

    def forward(self, logits, targets):
        ce = F.cross_entropy(
            logits.transpose(1, 2), targets, weight=self.class_weights,
            reduction="none", label_smoothing=self.label_smoothing,
        )
        return (ce * self.boundary_weights(targets)).mean()

def make_transformer_loss(config, class_weights):
    if config["kind"] == "weighted_ce":
        return TransformerWeightedCE(class_weights, config["label_smoothing"])
    if config["kind"] == "focal":
        return TransformerFocalLoss(
            class_weights, config["gamma"], config["label_smoothing"]
        )
    if config["kind"] == "boundary":
        return TransformerBoundaryLoss(
            class_weights, config["radius"], config["boundary_weight"],
            config["label_smoothing"],
        )
    raise ValueError(config)

## Recognition Transformer architecture

In [5]:
EMBED_DIM, NUM_HEADS, NUM_TRANSFORMER_LAYERS = 256, 8, 3
FEEDFORWARD_DIM, DROPOUT = 512, 0.2

class LinearProjector(nn.Module):
    def __init__(self, input_dim, embed_dim=EMBED_DIM, dropout=DROPOUT):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, embed_dim), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim), nn.ReLU(inplace=True), nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.proj(x)

class PositionalEncoding(nn.Module):
    def __init__(self, model_dim, dropout=DROPOUT, max_length=5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(max_length).unsqueeze(1)
        divisor = torch.exp(
            torch.arange(0, model_dim, 2) * (-math.log(10000.0) / model_dim)
        )
        encoding = torch.zeros(1, max_length, model_dim)
        encoding[0, :, 0::2] = torch.sin(position * divisor)
        encoding[0, :, 1::2] = torch.cos(position * divisor)
        self.register_buffer("pe", encoding)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class TemporalTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.pos_encoder = PositionalEncoding(EMBED_DIM, DROPOUT)
        layer = nn.TransformerEncoderLayer(
            d_model=EMBED_DIM, nhead=NUM_HEADS,
            dim_feedforward=FEEDFORWARD_DIM, dropout=DROPOUT,
            batch_first=True, activation="relu",
        )
        self.transformer = nn.TransformerEncoder(
            layer, num_layers=NUM_TRANSFORMER_LAYERS
        )
        self.norm = nn.LayerNorm(EMBED_DIM)
        self.classifier = nn.Linear(EMBED_DIM, NUM_CLASSES)

    def forward(self, x, padding_mask=None):
        x = self.pos_encoder(x)
        x = self.transformer(x, src_key_padding_mask=padding_mask)
        return self.classifier(self.norm(x))

class PhaseRecognizer(nn.Module):
    def __init__(self):
        super().__init__()
        self.projector = LinearProjector(FEATURE_DIM)
        self.temporal = TemporalTransformer()

    def forward(self, x, padding_mask=None):
        return self.temporal(self.projector(x), padding_mask=padding_mask)

def build_model():
    return PhaseRecognizer()

## Overlap-aware inference, training, and checkpoint recovery

In [6]:
MAX_EPOCHS, PATIENCE = 40, 5
LEARNING_RATE, WEIGHT_DECAY = 2e-4, 1e-2
GRADIENT_CLIP = 1.0

@torch.no_grad()
def predict_video(model, item):
    model.eval()
    features = item["features"]
    length = len(features)
    if length <= WINDOW_LENGTH:
        starts = [0]
    else:
        starts = list(range(0, length - WINDOW_LENGTH + 1, INFERENCE_WINDOW_STRIDE))
        final_start = length - WINDOW_LENGTH
        if starts[-1] != final_start:
            starts.append(final_start)

    logit_sum = torch.zeros(length, NUM_CLASSES, dtype=torch.float32)
    counts = torch.zeros(length, 1, dtype=torch.float32)
    for offset in range(0, len(starts), BATCH_SIZE):
        batch_starts = starts[offset:offset + BATCH_SIZE]
        windows, valid_lengths = [], []
        for start in batch_starts:
            window = features[start:start + WINDOW_LENGTH]
            valid = len(window)
            if valid < WINDOW_LENGTH:
                padding = torch.zeros(
                    WINDOW_LENGTH - valid, FEATURE_DIM, dtype=window.dtype
                )
                window = torch.cat([window, padding], dim=0)
            windows.append(window)
            valid_lengths.append(valid)
        batch = torch.stack(windows).to(DEVICE, non_blocking=True)
        positions = torch.arange(WINDOW_LENGTH, device=DEVICE).unsqueeze(0)
        valid_tensor = torch.tensor(valid_lengths, device=DEVICE).unsqueeze(1)
        padding_mask = positions >= valid_tensor
        with torch.autocast(
            device_type="cuda", dtype=torch.float16, enabled=USE_AMP
        ):
            logits = model(batch, padding_mask=padding_mask).float().cpu()
        for row, (start, valid) in enumerate(zip(batch_starts, valid_lengths)):
            logit_sum[start:start + valid] += logits[row, :valid]
            counts[start:start + valid] += 1
    if (counts == 0).any():
        raise RuntimeError("Inference windows did not cover the complete video")
    averaged_logits = logit_sum / counts
    return F.log_softmax(averaged_logits, dim=1).numpy()

@torch.no_grad()
def predict_dataset(model, dataset):
    results = {}
    start = time.perf_counter()
    for item in dataset.data:
        log_probabilities = predict_video(model, item)
        results[item["video_id"]] = {
            "labels": item["labels"].numpy(),
            "preds": log_probabilities.argmax(axis=1),
            "log_probs": log_probabilities,
        }
    return results, time.perf_counter() - start

@torch.no_grad()
def deterministic_window_loss(model, loader, criterion):
    model.eval()
    values = []
    for features, labels in loader:
        features = features.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.autocast(
            device_type="cuda", dtype=torch.float16, enabled=USE_AMP
        ):
            values.append(criterion(model(features), labels).item())
    return float(np.mean(values))

def train_or_load_candidate(feature_source, config, train_ds, val_ds,
                            class_weights, checkpoint_path):
    reset_seed()
    model = build_model().to(DEVICE)
    parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    criterion = make_transformer_loss(config, class_weights)
    pretrained = PRETRAINED_CHECKPOINTS.get(f"{feature_source}:{config['slug']}")
    history_columns = [
        "epoch", "train_loss", "val_loss", "val_accuracy", "val_f1_macro",
        "val_jaccard_macro", "val_edit", "val_f1_at_10", "val_f1_at_25",
        "val_f1_at_50", "val_f1_segmental_avg", "lr",
    ]
    if pretrained:
        pretrained = Path(pretrained)
        if not pretrained.exists():
            raise FileNotFoundError(pretrained)
        model.load_state_dict(load_checkpoint_state(pretrained), strict=True)
        torch.save({
            "model_state_dict": cpu_state_dict(model),
            "feature_source": feature_source, "candidate": config,
            "recovered_from": str(pretrained),
        }, checkpoint_path)
        return model, pd.DataFrame(columns=history_columns), {
            "parameters": parameter_count, "training_seconds": np.nan,
            "peak_vram_mb": np.nan, "best_epoch": np.nan, "recovered": True,
        }
    if not TRAIN_IF_CHECKPOINT_MISSING:
        raise FileNotFoundError(
            f"No checkpoint mapped for {feature_source}:{config['slug']}"
        )

    train_windows = TransformerWindowDataset(train_ds, train=True)
    val_windows = TransformerWindowDataset(val_ds, train=False)
    sample_weights = window_sample_weights(
        train_windows, class_weights.detach().cpu()
    )
    sampler = WeightedRandomSampler(
        sample_weights, num_samples=len(sample_weights), replacement=True
    )
    train_loader = DataLoader(
        train_windows, batch_size=BATCH_SIZE, sampler=sampler,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    val_loader = DataLoader(
        val_windows, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-5
    )
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    history, best_primary, best_epoch, stale = [], -np.inf, None, 0
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    training_start = time.perf_counter()

    print(f"\n===== {feature_source} / {config['slug']} =====")
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        losses = []
        for features, labels in train_loader:
            features = features.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            features = features + torch.randn_like(features) * 0.01
            if torch.rand(1).item() < 0.3:
                features = features * (torch.rand_like(features) > 0.05)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type="cuda", dtype=torch.float16, enabled=USE_AMP
            ):
                loss = criterion(model(features), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            scaler.step(optimizer)
            scaler.update()
            losses.append(loss.item())

        val_results, _ = predict_dataset(model, val_ds)
        val_metrics = evaluate_sequences(val_results)
        val_loss_value = deterministic_window_loss(model, val_loader, criterion)
        scheduler.step(epoch)
        primary = val_metrics["f1_segmental_avg"]
        history.append({
            "epoch": epoch, "train_loss": float(np.mean(losses)),
            "val_loss": val_loss_value, "val_accuracy": val_metrics["accuracy"],
            "val_f1_macro": val_metrics["f1_macro"],
            "val_jaccard_macro": val_metrics["jaccard_macro"],
            "val_edit": val_metrics["edit"],
            "val_f1_at_10": val_metrics["f1_at_10"],
            "val_f1_at_25": val_metrics["f1_at_25"],
            "val_f1_at_50": val_metrics["f1_at_50"],
            "val_f1_segmental_avg": primary,
            "lr": optimizer.param_groups[0]["lr"],
        })
        if primary > best_primary:
            best_primary, best_epoch, stale = primary, epoch, 0
            torch.save({
                "model_state_dict": cpu_state_dict(model),
                "feature_source": feature_source, "candidate": config,
                "epoch": epoch,
                "best_validation_f1_segmental_avg_raw": best_primary,
            }, checkpoint_path)
        else:
            stale += 1
        print(
            f"epoch={epoch:03d} train={np.mean(losses):.4f} val={val_loss_value:.4f} "
            f"Acc={val_metrics['accuracy']:.3f} Edit={val_metrics['edit']:.3f} "
            f"F1avg={primary:.3f} stale={stale}/{PATIENCE}"
        )
        if stale >= PATIENCE:
            break

    model.load_state_dict(load_checkpoint_state(checkpoint_path), strict=True)
    return model, pd.DataFrame(history), {
        "parameters": parameter_count,
        "training_seconds": time.perf_counter() - training_start,
        "peak_vram_mb": torch.cuda.max_memory_allocated() / (1024 ** 2),
        "best_epoch": best_epoch, "recovered": False,
    }

## Run all feature sources and create standardized archives

In [7]:
MODEL_NAME = "Recognition Transformer"
MODEL_SLUG = "recognition_transformer"
archives = []

for feature_source, feature_dir in RESOLVED_FEATURE_DIRS.items():
    run_name = f"{MODEL_SLUG}_{feature_source}"
    results_dir = OUTPUT_ROOT / run_name
    if results_dir.exists():
        shutil.rmtree(results_dir)
    checkpoint_dir = results_dir / "checkpoints"
    prediction_dir = results_dir / "predictions"
    candidate_dir = results_dir / "candidate_selection"
    for directory in (results_dir, checkpoint_dir, prediction_dir, candidate_dir):
        directory.mkdir(parents=True, exist_ok=True)

    inventory = inspect_feature_source(feature_dir)
    inventory.to_csv(results_dir / "feature_inventory.csv", index=False)
    feature_mean, feature_std = (
        calculate_train_stats(feature_dir, TRAIN_IDS)
        if NORMALIZE_FEATURES else (None, None)
    )
    train_ds = FeatureSequenceDataset(
        feature_dir, TRAIN_IDS, feature_mean, feature_std
    )
    val_ds = FeatureSequenceDataset(feature_dir, VAL_IDS, feature_mean, feature_std)
    class_weights, class_counts = balanced_class_weights(train_ds)
    transition_log, start_log = training_transition_model(train_ds)

    candidate_rows = []
    for config in LOSS_CANDIDATES:
        slug = config["slug"]
        checkpoint_path = checkpoint_dir / f"best_{slug}.pt"
        model, history, summary = train_or_load_candidate(
            feature_source, config, train_ds, val_ds, class_weights, checkpoint_path
        )
        history_path = candidate_dir / f"training_history_{slug}.csv"
        curve_path = candidate_dir / f"training_curves_{slug}.png"
        history.to_csv(history_path, index=False)
        plot_training_history(history, curve_path)

        validation_results, validation_seconds = predict_dataset(model, val_ds)
        validation_results = add_postprocessing(
            validation_results, transition_log, start_log
        )
        validation_table = validation_postprocessing_table(validation_results)
        validation_table.to_csv(
            candidate_dir / f"validation_postprocessing_{slug}.csv", index=False
        )
        selected_pp = validation_table.iloc[0]["postprocessing"]
        selected_metrics = validation_table.iloc[0].to_dict()
        candidate_rows.append({
            "candidate": slug,
            "postprocessing": selected_pp,
            "parameters": summary["parameters"],
            "training_minutes": summary["training_seconds"] / 60,
            "peak_vram_mb": summary["peak_vram_mb"],
            "validation_inference_seconds": validation_seconds,
            "best_epoch": summary["best_epoch"],
            "recovered_checkpoint": summary["recovered"],
            **{key: value for key, value in selected_metrics.items()
               if key != "postprocessing"},
        })
        del model, validation_results
        gc.collect()
        torch.cuda.empty_cache()

    candidate_table = pd.DataFrame(candidate_rows).sort_values(
        ["f1_segmental_avg", "edit", "f1_macro"], ascending=False
    ).reset_index(drop=True)
    candidate_table.to_csv(results_dir / "candidate_validation_summary.csv", index=False)
    selected_candidate = candidate_table.iloc[0]["candidate"]
    selected_postprocessing = candidate_table.iloc[0]["postprocessing"]
    selected_key = "preds" if selected_postprocessing == "raw" else "viterbi"
    selected_checkpoint = checkpoint_dir / f"best_{selected_candidate}.pt"

    shutil.copy2(
        candidate_dir / f"training_history_{selected_candidate}.csv",
        results_dir / "training_history.csv",
    )
    shutil.copy2(
        candidate_dir / f"training_curves_{selected_candidate}.png",
        results_dir / "training_curves.png",
    )
    shutil.copy2(
        candidate_dir / f"validation_postprocessing_{selected_candidate}.csv",
        results_dir / "validation_postprocessing.csv",
    )
    shutil.copy2(selected_checkpoint, checkpoint_dir / f"best_{MODEL_SLUG}.pt")

    model = build_model().to(DEVICE)
    model.load_state_dict(load_checkpoint_state(selected_checkpoint), strict=True)
    validation_results, validation_inference_seconds = predict_dataset(model, val_ds)
    validation_results = add_postprocessing(validation_results, transition_log, start_log)
    for video_id, values in validation_results.items():
        np.savez_compressed(
            prediction_dir / f"{run_name}_val_video{video_id:02d}.npz",
            labels=values["labels"], log_probs=values["log_probs"],
            raw=values["preds"], viterbi=values["viterbi"],
        )

    # The test set is instantiated only after candidate/checkpoint/post-processing freeze.
    test_ds = FeatureSequenceDataset(feature_dir, TEST_IDS, feature_mean, feature_std)
    test_results, test_inference_seconds = predict_dataset(model, test_ds)
    test_results = add_postprocessing(test_results, transition_log, start_log)
    test_metrics = evaluate_sequences(test_results, selected_key)
    per_video = per_video_metrics(test_results, selected_key)

    selected_row = candidate_table.iloc[0]
    final_metrics = pd.DataFrame([{
        "model": MODEL_NAME,
        "feature_source": feature_source,
        "feature_lineage": FEATURE_SOURCES[feature_source]["description"],
        "postprocessing": selected_postprocessing,
        "parameters": int(selected_row["parameters"]),
        "training_minutes": selected_row["training_minutes"],
        "peak_vram_mb": selected_row["peak_vram_mb"],
        "validation_inference_seconds": validation_inference_seconds,
        "test_inference_seconds": test_inference_seconds,
        **test_metrics,
    }])
    final_metrics.to_csv(results_dir / "final_metrics.csv", index=False)
    recurrent_columns = [
        "model", "feature_source", "postprocessing", "parameters",
        "training_minutes", "peak_vram_mb", "test_inference_seconds",
        "accuracy", "precision_macro", "recall_macro", "f1_macro",
        "jaccard_macro", "edit", "f1_at_10", "f1_at_25", "f1_at_50",
        "f1_segmental_avg",
    ]
    final_metrics[recurrent_columns].to_csv(
        results_dir / "final_model_comparison.csv", index=False
    )
    per_video.to_csv(results_dir / "test_metrics_per_video.csv", index=False)

    for video_id, values in test_results.items():
        np.savez_compressed(
            prediction_dir / f"{run_name}_test_video{video_id:02d}.npz",
            labels=values["labels"], log_probs=values["log_probs"],
            raw=values["preds"], viterbi=values["viterbi"],
            selected=values[selected_key],
        )
    save_confusion_matrix(
        test_results, selected_key,
        f"{MODEL_NAME} — {feature_source} features",
        results_dir / "confusion_matrix.png",
    )

    protocol = {
        "implementation": "standardized sliding-window recognition Transformer experiment",
        "seed": SEED,
        "feature_source": feature_source,
        "feature_lineage": FEATURE_SOURCES[feature_source]["description"],
        "feature_directory": str(feature_dir),
        "split": {"train": TRAIN_IDS, "validation": VAL_IDS, "test": TEST_IDS},
        "model": {
            "window_length": WINDOW_LENGTH, "training_stride": TRAIN_WINDOW_STRIDE,
            "inference_stride": INFERENCE_WINDOW_STRIDE, "embedding_dim": EMBED_DIM,
            "attention_heads": NUM_HEADS, "transformer_layers": NUM_TRANSFORMER_LAYERS,
            "feedforward_dim": FEEDFORWARD_DIM, "dropout": DROPOUT,
            "input_dim": FEATURE_DIM,
        },
        "candidate_losses": LOSS_CANDIDATES,
        "selected_candidate": selected_candidate,
        "selected_postprocessing": selected_postprocessing,
        "training": {
            "max_epochs": MAX_EPOCHS, "patience": PATIENCE,
            "optimizer": "AdamW", "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY, "scheduler": "CosineAnnealingWarmRestarts",
            "batch_size": BATCH_SIZE,
            "class_weights": class_weights.detach().cpu().tolist(),
            "class_counts": class_counts.tolist(),
            "normalization": NORMALIZE_FEATURES, "amp": USE_AMP,
        },
        "selection_rule": (
            "epoch, loss candidate, and raw/Viterbi selected only by validation "
            "average F1@10/25/50 (Edit and macro-F1 tie-breakers); test evaluated once"
        ),
        "aggregation_note": (
            "accuracy is pooled frames; segmental F1 counts are pooled across videos; "
            "Edit is the mean of per-video normalized Edit scores"
        ),
    }
    with open(results_dir / "protocol.json", "w") as file:
        json.dump(protocol, file, indent=2)

    archive = make_archive(results_dir, run_name)
    archives.append(archive)
    print("Saved:", archive)

    del model, train_ds, val_ds, test_ds, validation_results, test_results
    gc.collect()
    torch.cuda.empty_cache()

display(pd.DataFrame({"archive": [str(path) for path in archives]}))


===== teammates / weighted_ce =====
epoch=001 train=0.1664 val=1.1121 Acc=0.802 Edit=0.241 F1avg=0.293 stale=0/5
epoch=002 train=0.0177 val=1.5258 Acc=0.775 Edit=0.154 F1avg=0.198 stale=1/5
epoch=003 train=0.0097 val=1.4205 Acc=0.799 Edit=0.169 F1avg=0.229 stale=2/5
epoch=004 train=0.0065 val=1.5585 Acc=0.799 Edit=0.191 F1avg=0.243 stale=3/5
epoch=005 train=0.0033 val=1.6385 Acc=0.798 Edit=0.231 F1avg=0.280 stale=4/5
epoch=006 train=0.0021 val=1.9050 Acc=0.775 Edit=0.188 F1avg=0.247 stale=5/5

===== teammates / focal_g2 =====
epoch=001 train=0.4889 val=0.8146 Acc=0.782 Edit=0.175 F1avg=0.230 stale=0/5
epoch=002 train=0.2683 val=0.9643 Acc=0.686 Edit=0.143 F1avg=0.188 stale=1/5
epoch=003 train=0.2580 val=0.8792 Acc=0.792 Edit=0.230 F1avg=0.286 stale=0/5
epoch=004 train=0.2537 val=0.8922 Acc=0.797 Edit=0.198 F1avg=0.263 stale=1/5
epoch=005 train=0.2513 val=0.8083 Acc=0.797 Edit=0.191 F1avg=0.245 stale=2/5
epoch=006 train=0.2501 val=0.8299 Acc=0.783 Edit=0.187 F1avg=0.227 stale=3/5
epoch

,archive
0,/kaggle/working/recognition_transformer_teamma...
1,/kaggle/working/recognition_transformer_finetu...
2,/kaggle/working/recognition_transformer_origin...
